In [9]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
import yfinance as yf
from ddgs import DDGS

# setting up local LLM(instead of using OpenAI's API(I have no money))
local_llm = LLM(
    model="ollama/llama3.1", 
    base_url="http://localhost:11434"
)

# 2. BUILDING THE TOOLS (Giving AI hands)

@tool("Fetch Stock Price")
def fetch_stock_price(ticker: str) -> str:
    """Fetches the 30-day stock price history for a given ticker."""
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period="1mo")
        
        if hist.empty:
            return f"Action Failed: The ticker '{ticker}' is not recognized or has no data. Do not invent data. State that the stock is invalid."
            
        data_summary = f"Recent price data for {ticker}:\n"
        for index, row in hist.iterrows():
            data_summary += f"{index.date()}: Close = ${row['Close']:.2f}\n"
            
        return data_summary
        
    except Exception as e:
        return f"Action Failed: An error occurred while fetching data for {ticker}. Error: {str(e)}"

@tool("News Scraper")
def fetch_stock_news(ticker: str) -> str:
    """Searches the web for the latest financial news about a given stock ticker."""
    print(f"\n[TOOL CALLED] Searching the web for {ticker} news...")
    results = DDGS().text(f"latest financial news {ticker} stock", max_results=3)
    news_summary = "\n".join([f"- {res['title']}: {res['body']}" for res in results])
    return f"Latest news for {ticker}:\n{news_summary}"




In [10]:
# Test the tool with Tesla
tesla_news = fetch_stock_news.run("TSLA")
print(tesla_news)



[TOOL CALLED] Searching the web for TSLA news...
Latest news for TSLA:
- TSLA Stock Performance & Market Trends | Stocks.News: ThelatestTechnical Analysis Summary onStocks.Newsoffers insights into market trends using key Technical Indicators like RSI and MACD, alongside ...
- TSLA Stock: Price Chart Analysis | Market Pulse: Subscribe now to our mailing list and receive thelatestmarketnewsand insights delivered directly to your inbox.
- TSLA Stock: Report Pending | Market Pulse: TSLAshare price rises on thenewsthat the company has started production of the Cybertruck model. ... and corporatenewsshaping thefinancial...


In [3]:

# 3. DEFINING THE AGENTS

quant = Agent(
    role="Senior Data Fetcher",
    goal="Accurately retrieve and summarize mathematical stock data for {ticker}.",
    backstory="You are a ruthless, numbers-driven Wall Street quant. You only care about the raw data and 30-day trends.",
    tools=[fetch_stock_price],
    llm=local_llm,
    verbose=True
)

analyst = Agent(
    role="News & Media Analyst",
    goal="Gauge public perception and find potential catalysts for {ticker} based on recent news.",
    backstory="You are an expert at reading between the lines of financial news. You identify if the media sentiment is bullish (positive) or bearish (negative).",
    tools=[fetch_stock_news],
    llm=local_llm,
    verbose=True
)

cio = Agent(
    role="Lead Portfolio Manager",
    goal="Synthesize data and news to write a final, structured investment memo for {ticker}.",
    backstory="You are a cautious but decisive hedge fund manager. You take the raw data from your Quant and the sentiment from your Analyst to make a final Buy, Hold, or Sell recommendation.",
    llm=local_llm,
    verbose=True
)
# 4. DEFINING THE TASKS 
task_price = Task(
    description="Use your tool to fetch the 30-day price history for {ticker}. Format it into a clean summary indicating if the trend is up, down, or flat.",
    expected_output="A short paragraph summarizing the 30-day price trend.",
    agent=quant
)

task_news = Task(
    description="Search the web for the latest news on {ticker}. Summarize the top events and declare the overall sentiment (Bullish, Bearish, or Neutral).",
    expected_output="A bulleted list of news summaries and a final sentiment verdict.",
    agent=analyst
)

task_memo = Task(
    description="Review the price data from the Quant and the news sentiment from the Analyst. Write a 3-paragraph investment memo concluding with a definitive Buy, Hold, or Sell rating for {ticker}.",
    expected_output="A 3-paragraph investment memo with a clear final recommendation.",
    agent=cio
)

# 5. FORMING THE CREW AND RUNNING IT
quant_crew = Crew(
    agents=[quant, analyst, cio],
    tasks=[task_price, task_news, task_memo],
    verbose=True
)

if __name__ == "__main__":
    target_ticker = input("Enter a company name or stock ticker to analyze: ").strip()
    
    print(f"\n🚀 Kicking off research on {target_ticker}...\n")
    result = quant_crew.kickoff(inputs={'ticker': target_ticker})
    
    print("\n==========================================")
    print(f"FINAL INVESTMENT MEMO FOR {target_ticker}:")
    print("==========================================")
    print(result)

    # SAVING THE OUTPUT TO A FILE
    filename = f"{target_ticker}_investment_memo.md"
    
    with open(filename, "w", encoding="utf-8") as file:
        file.write(f"# Investment Memo: {target_ticker}\n\n")
        file.write(str(result))
        
    print(f"\n The memo has been saved to your project folder as: {filename}")

c:\Users\mahdi\OneDrive\Desktop\Quant Team\.venv\Lib\site-packages\crewai\agent\core.py:301: DeprecationWarning: deprecated
  if self.reasoning and self.planning_config is None:
c:\Users\mahdi\OneDrive\Desktop\Quant Team\.venv\Lib\site-packages\crewai\agent\core.py:301: DeprecationWarning: deprecated
  if self.reasoning and self.planning_config is None:
c:\Users\mahdi\OneDrive\Desktop\Quant Team\.venv\Lib\site-packages\crewai\agent\core.py:301: DeprecationWarning: deprecated
  if self.reasoning and self.planning_config is None:



🚀 Kicking off research on apple...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b27dd162-fbff-4661-98be-5b6c7f1f6dcc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

c:\Users\mahdi\OneDrive\Desktop\Quant Team\.venv\Lib\site-packages\crewai\crew.py:1312: DeprecationWarning: deprecated
  and hasattr(agent, "multimodal")
c:\Users\mahdi\OneDrive\Desktop\Quant Team\.venv\Lib\site-packages\crewai\crew.py:1313: DeprecationWarning: deprecated
  and getattr(agent, "multimodal", False)


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use your tool to fetch the 30-day price history for apple. Format it into a clean summary indicating if  │
│  the trend is up, down, or flat.                                                                                │
│  ID: 7dddc52c-728b-472c-b235-13ebf31199b4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Portfolio Manager                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Investment Memo for Apple Inc.**                                                                             │
│                                                                                                                 │
│  Over the past 30 days, our quantitative analysis has provided a clear picture of Apple's stock performance.    │
│  The data shows that over this period, Apple shares have traded in a relatively narrow range, with prices       │
│  ranging between $264.72 and $263.75. This stability suggests that investors are maintaining their faith in     │
│  the company's prospects, but not necessarily getting overly enthusiastic about future growth.                  │
│                                                                                                                 │
│  Meanwhile, our analyst has been monitoring sentiment from various news sources and providing input on broader  │
│  market trends. While there have been some volatility-increasing comments on supply chain constraints for       │
│  upcoming product releases, overall market sentiment remains cautiously optimistic about Apple's long-term      │
│  prospects. Notably, investors continue to focus on the company's strong fundamentals, particularly its         │
│  market-leading positions in key product categories such as smartphones and personal computers.                 │
│                                                                                                                 │
│  In light of these findings, our team recommends a Buy rating for Apple shares. Despite short-term volatility   │
│  related to supply chain concerns, the underlying momentum and stable fundamental backdrop justify taking a     │
│  long position. Moreover, with the stock trading at relatively reasonable valuations compared to its long-term  │
│  median, we believe there is ample room for potential upside over the coming quarters.                          │
│                                                                                                                 │
│  **Final Recommendation:** Buy                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the price data from the Quant and the news sentiment from the Analyst. Write a 3-paragraph        │
│  investment memo concluding with a definitive Buy, Hold, or Sell rating for apple.                              │
│  Agent: Lead Portfolio Manager                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b27dd162-fbff-4661-98be-5b6c7f1f6dcc                                                                       │
│  Final Output: **Investment Memo for Apple Inc.**                                                               │
│                                                                                                                 │
│  Over the past 30 days, our quantitative analysis has provided a clear picture of Apple's stock performance.    │
│  The data shows that over this period, Apple shares have traded in a relatively narrow range, with prices       │
│  ranging between $264.72 and $263.75. This stability suggests that investors are maintaining their faith in     │
│  the company's prospects, but not necessarily getting overly enthusiastic about future growth.                  │
│                                                                                                                 │
│  Meanwhile, our analyst has been monitoring sentiment from various news sources and providing input on broader  │
│  market trends. While there have been some volatility-increasing comments on supply chain constraints for       │
│  upcoming product releases, overall market sentiment remains cautiously optimistic about Apple's long-term      │
│  prospects. Notably, investors continue to focus on the company's strong fundamentals, particularly its         │
│  market-leading positions in key product categories such as smartphones and personal computers.                 │
│                                                                                                                 │
│  In light of these findings, our team recommends a Buy rating for Apple shares. Despite short-term volatility   │
│  related to supply chain concerns, the underlying momentum and stable fundamental backdrop justify taking a     │
│  long position. Moreover, with the stock trading at relatively reasonable valuations compared to its long-term  │
│  median, we believe there is ample room for potential upside over the coming quarters.                          │
│                                                                                                                 │
│  **Final Recommendation:** Buy                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL INVESTMENT MEMO FOR apple:
**Investment Memo for Apple Inc.**

Over the past 30 days, our quantitative analysis has provided a clear picture of Apple's stock performance. The data shows that over this period, Apple shares have traded in a relatively narrow range, with prices ranging between $264.72 and $263.75. This stability suggests that investors are maintaining their faith in the company's prospects, but not necessarily getting overly enthusiastic about future growth.

Meanwhile, our analyst has been monitoring sentiment from various news sources and providing input on broader market trends. While there have been some volatility-increasing comments on supply chain constraints for upcoming product releases, overall market sentiment remains cautiously optimistic about Apple's long-term prospects. Notably, investors continue to focus on the company's strong fundamentals, particularly its market-leading positions in key product categories such as smartphones and personal compute

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯